# 8교시. 실무 적용 시나리오 설계 및 최종 정리

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/08_business_application.ipynb)

**이번 교시 행동:** 견적서·신청서·거래명세서 실물 사진을 비교하고 첫 PoC 한 가지를 고릅니다.

**통과 증거:** `course_outputs/poc_candidate_card.md`

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

화면의 실행 모드를 먼저 확인합니다.

- `LIVE`: 현재 파일에 실제 모델을 실행한 결과
- `PREPARED_FALLBACK`: 공개 샘플을 사람이 검수해 둔 복구 결과
- 3분 이상 멈추면 실행을 중지하고 복구 결과로 계속합니다.
- 각 교시 끝에서 `CHECKPOINT PASS`와 산출물 파일을 확인합니다.


In [ ]:
import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
VALIDATION_MODE = os.getenv("COURSE_VALIDATE_PREPARED") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or VALIDATION_MODE:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 준비 입력을 쓰려면 "
            "USE_PREPARED_INPUT=True로 바꾸세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if VALIDATION_MODE:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())


In [ ]:
import base64
import io
from PIL import Image
try:
    from IPython.display import display
except ImportError:
    display = lambda image: None

EXTENSION_IMAGES = {'quotation': '...', 'application': '...', 'transaction_statement': '...'}
EXTENSION_EXAMPLES = {'quotation': {'name': '견적서', 'fields': ['문서번호', '공급자', '수신', '견적일', '품목', '총액'], 'rules': ['수량×단가=품목금액', '공급가액+부가세=총액'], 'risk': '총액 오류는 구매 의사결정에 직접 영향'}, 'application': {'name': '신청서', 'fields': ['신청번호', '신청자', '소속', '신청 과정', '승인'], 'rules': ['필수 동의', '관리자 승인 상태'], 'risk': '개인정보와 승인 누락을 사람이 확인'}, 'transaction_statement': {'name': '거래명세서', 'fields': ['문서번호', '공급자', '거래일', '품목', '세액', '총액'], 'rules': ['품목 합계=공급가액', '공급가액+세액=총액'], 'risk': '표 행·열 대응이 어긋나면 정산 오류'}}
for key, payload in EXTENSION_IMAGES.items():
    image = Image.open(io.BytesIO(base64.b64decode(payload))).convert("RGB")
    image.thumbnail((320, 400))
    print(key, image.size)
    display(image)


## 형식이 바뀌면 생기는 어려움

- **Excel**: 수식, 병합 셀, 숨김 시트, 숫자 서식
- **Word**: 머리글, 텍스트박스, 변경 추적, 이미지로 삽입된 본문
- **PDF**: 텍스트·스캔 혼합 페이지, 암호, 깨진 문자맵
- **PPT**: 그룹 도형, 읽기 순서, 발표자 노트
- **표 캡처**: 셀 관계가 사라져 행·열 위상을 다시 복원해야 함


In [ ]:
import re
import zipfile

OFFICE_FILES = {'quotation.xlsx': 'UEsDBBQAAAAIAIG0/FxPfYNzHQEAAFACAAAPAAAAeGwvd29ya2Jvb2sueG1stdLLSgMxFAbgVwnZO5mbvQydduPGrS9QMslJJ3SSDEmqs9WV4E4sFKEguFLqxrW+kE7fQazSFlduujucHw4f/GcwalSFzsE6aXSOoyDECDQzXOpJjmdeHPXwaDhosgtjp4UxU9SoSrusyXHpfZ0R4lgJirrA1KAbVQljFfUuMHZCXG2BclcCeFWROAw7RFGp8fe9zdZtJ6Spghx/vK7ah8txe3/Xzt8x2kSnPMcRRjaTPMdnIkko64ZJygRP+90U/4Lsf0BGCMngxLCZAu1/RBYq6qXRrpS1w4j8Ja1vrz+fn/Ys8Z6l6PeBdyCBIu3S3uEti3l7sxy3j2/tctFerdbzlz1YsoWlPArjiLNY0CKlx3AAGNlVSHbfMfwCUEsDBBQAAAAIAIG0/Fyo7/cH1wIAACMbAAANAAAAeGwvc3R5bGVzLnhtbOVZ3W6bMBR+FeRKu1rLTwKkXWmVpkHaxXqTTtqtAyax5B9mnA76FLvfK+4hJsAEUiUpXX5I1dxgH3G+88X+7HNycn2bUqI9IZFgzjxgXhhAQyzgIWYzDyxkdD4AtzfX6VUiM4Imc4SkllLCkqvUA3Mp4ytdT4I5ojC54DFiKSURFxTK5IKLmZ7EAsEwyd0o0S3DcHQKMQM5IltQn8pEC/iCSQ9YDaNWPr6GHrAMA2gl5IiHyANZlmXnlJ6HIdD0DS7mqsvZ57Mz49PPBZdf/v75XQ4KZ33JIgeKOKvpmCWf3FZ8/WftCRIPmGYVFVJUmkZQECx5BVh5VM9p+f4SYKAAAk640MRs6gFffXaFNtdBm27vznYPAr0f1v110IZj+33nrdC4DWt3aA9zUR1vG4cCQ6J9ZzjgIdK+TTbF2CKythC7SGKvMXZcil1E0jbGLmrZHkMNylsFE7K8VdzyUsGE5M8YSokE8zEhmho/ZjHyAOMMLRHVy686zQTMTMt+s1/CCQ5LXrPRBpnoK/57wh8PxmPfPhy+4buOMz4g/5F/7/cOh+/7vnM/2I6vBoXSplyESNQZDNRGhaHeKESOCJnkOf1H9MIjjRq5tEi+bDnEhKhhCaUmJXoTsgrRQO9Z/wufRnWcVgDWJgAYxyR7WNApEn5RIORfurT6nDVnmJB6dleAFfOtFMyPQqGYDwmeMYpq5cDKkBeUEgf5RRkgJpGoJJJGLXew19HyWd1TaHMQDkFBVdrdklhPod8Rhd5HofDaeZ5zgZ85kzucaLujBex3REH9Cu2WxPqtcDqiYH8UCq8dp18Cxo8oLaHanKNeR4XFiohPICccZx32vn3HWbm903becS3pdi/WQfe15OX7qx5WrhyzqwJipRg+AS3lzan3mLguTyFxncD+HWcd9r597iknLtVyanSbiu7Ti3bW0q7l3VsPPOQcyWpTqdm8Sopp/XfXzT9QSwMEFAAAAAgAgbT8XPpcAVkDAwAA2g0AABMAAAB4bC90aGVtZS90aGVtZTEueG1svVfbcpswFPwVRu8NN3PzhGQSx24f0mmnyQ/IIECNEB5Jjp2/7yBuAozjNHbsB0tiz9lF57DC17f7nGiviHFc0BCYVwbQEI2KGNM0BFuRfPPB7c01nIsM5UijMEchWGRQfP/9DLR9TiifwxBkQmzmus6jDOWQXxUbRPc5SQqWQ8GvCpbqMYM7TNOc6JZhuHoOMQVt3iVBOaKClwsRYU/RAbLyWvxilj/8jS8I014hCcEO07jYPaO9ABqBXCwIC4EhP0DTb671NoqIiWAlcCU/TWAdEb9YMpCl6zbSWFr+zOwYJIKIMXDpl98uo0TAKEK0lqOCTcc1fKsBK6hqeCB74Jn2IEBhsMcMgXtvzfoBElUNZ+MbXQXLB6cfIFHV0BkF3BnWfWD3AySqGrqjgNnyzrOW/QCJygimL2O46/m+28BbTFKQHwfxgesa3kOD72C60mpVAip6jfcrSXCEZN/l8G/BVgUVsspQYKqJtw1KYFQ2KCR4zbD2iNNMSB44R/AdQMSPAvQBZ47puwKOUB8hbek6Bl3dDLk1uZh8JBNMyJN4I+iRS3G8IDheYULkREa1pdhkC8Iawh4wZbAb8zpVyrVNwUNggMlc0kEwFdWa6zVPPZyTbf6ziOumN1s7gHMORXfBcBSfaBnkLOWqhhJ3sg7PntDR0Q112CfqkHdyshDf/LCQ4KgQXSkPwVSD5SnhzGq75REkKC4LVifolfUsJQ5mU3dkfXZrTygxz2CMmrzGlJKpZuu68AxFVqR4/mElQTAhpNyqSxRZH9sBof2Ztiv5vebu/sssNoyLB8izCicvtecrVWgCw/kCGqvcmcvR6MM9REmCIjGx0k0fuaizHLz8WXQ5KbYCsacs3mlrsmV/YBwCxzMdA2gx5qIpgBZj1rXP+P2iW4dkk8HayXsPbYWX45ZTESvlDKX357Xidbo6y3H1ftTAtabs1pt+Ei9wPgbKuaT4R+B/1FMrqzz3sanqUOVNGq09Ic++kNF2Xfl1hjps2dJjm9cxORv8gWpWbv4BUEsDBBQAAAAIAIG0/FwNHrnoZQAAAHMAAAAUAAAAeGwvc2hhcmVkU3RyaW5ncy54bWwFwVEKwyAMANCrSP5n3D7GkNqeRdq0CiYWkw2Pv/eWbXJzPxpauyR4+gCOZO9HlSvB187HB7Z1mVHV3OQmGmeCYnZHRN0LcVbfb5LJ7eyDs6nv40K9B+VDC5Fxw1cIb+RcBRyuf1BLAwQUAAAACACBtPxce2eF/OQDAAB9DAAAGAAAAHhsL3dvcmtzaGVldHMvc2hlZXQxLnhtbJ1XXW/iRhT9K9ORqu2Hgo2/ICiw6oLdVtpV263aPnthAGsxRrYTeGQVumJJW6VVUNOsqYiUKptqpdJs0nql9KU/xx7/h8o2AQMXpJQXM2fm3HvPuWZm2Lnf0Rtoj5iWZjTzOJ1iMSLNslHRmrU83rWrW1l8v7DTybUN86lVJ8RGHb3RtHKdPK7bdivHMFa5TnTVShkt0uzojaph6qptpQyzxlgtk6iViKY3GI5lJUZXtSYOA0bo1xppWwsjZNWN9semVnmoNYmVxyxGYeonhvE0nP60EkFMYYcBQyhR9s9NVCFVdbdhPzbanxCtVrfzOC1GvE6ubDQiQtloIF0LVWOkq53o2dYqdj2P0xJGda1SIc0oXXnXsg39m+ncPExM56Z0bkbnhDvQ+Smdn9H5O7CFKVuY1565A12c0sX/R5emdOkOdGbegahlJdVWw4FptJEZLQq7xQu35Fn/otemHK75KI2RlcdcGiM7jy3bjKb2Ct7lazp+RnsO8i67wfMxoidHdPhPmHIvTjwL8WAegpmBRQgsQaAMgcoCyESSEsq4hAAuXskl6Q8gsAiBJQiUIVBZAFdK4hMl8fFKfslT/7VLe45/2Q+OXdDIKY9b4n2xxbGctMVmuOwWm4aYRR6SNgWF5da+ufbcF96kS4enUDB5Dc//qxuSemDtyhoSvRotp1mxTkhYJ8DWxTXTXw9B3wTYt2Do+G9f0uOz4OiGDs5A4wTIuCkoRvHiPbZa+PKrR+/dC37q+79f3HtH5nIy/34YqxrnEtnwA9oJRisJH6TZd5cDgM6u4X8oC0n6Kn/FZzHhswj7TPvH9GAMmizCJtP9vn/V89xJcPI96LAIOQyBMgQqC+CKJCkhSVrz6sQ72egGVDUlSQlz9wqCxPHw70yCxECgDIGKtFFMJiEms6Y/zjg4eeW5E2/SA/Vk1ugRRFBPBtIDgTIEKpmNerIJPdl1zQkPF7r/LNh3QD1Z+K3zrg+pc0pPLlAwvKC9P1G8t4Ias5BGCJQhUMlu1Lid0LgNnUQQWITAEgTKEKhsbywpzcbnvsBtPPfZOEhmedP8waXnPyJv0vUHvwVDB9GXh/4bF9Fvu//+HW0Po/D5ynvbRfTACQauPzhCdHTl/+HS8y76rPjYP/+OOjf+qYOCX4Z05AbDMaL9Y0R/fk4H1/5B3z84S8E3iXlRyasEhJZAVAZRZRG9dez23nt7bdKJWSNF0ohvVLMRMkk1NCwXXkqioCtTbC5KEcdeDNNSa+SRata0poUapGrnMZvKYGTGHYm+20Yr+iZi9MSwbUO/HdWJWiFmOOIxqhqGPRvEmWb/JAr/AVBLAwQUAAAACACBtPxcyX3nr24CAADNBgAAGAAAAHhsL3dvcmtzaGVldHMvc2hlZXQyLnhtbJ2VSYvbMBSA/4rQcaDxlm1C7KFZukALwxzas2LLsRjbCpKS+Nhj2hlo6QIzQ9PSW6enQnOY3zRx/kOxnHUqQ5OT9aT3vSd9YKl5kkQhGGHGCY1taJR0CHDsUo/EfRsOhf+oDk+cZtIYU3bOA4wFSKIw5o3EhoEQg4amcTfAEeIlOsBxEoU+ZRESvERZX+MDhpEnsSjUTF2vahEiMcwKytlXBI/5TgR4QMdPGfFekBhzG+oQZK17lJ5ny889OaU5TU1Z4onsfsqAh300DMUZHT/DpB8IGxoVySUNl4YScGkIIpKdGoIIJfI7Jp4IbGhWIQiI5+FYtnOHXNDodb5mbMrkuLnEzTVu6Hvg1hK3Nvg+3ctLvHwYXlnilT1wbeNQSu8ggbKA0TFgWZLskA0fGxBwG5p1CIQNuWByaeQsPk7mv26zQqO83JpoFRDp5Gr+/a2KaBcQ84uf97/fqIhOcY/0YqoiugXE/d0k/fJjh9Ckhy0d5pYOMy9jPtRxc5lOr9KbW5B9vn1Qitlh839o5JSVRpapxzup9Yqu60odinTfaZlHbTNL9HPcKusFfPdQ/h9V1pYqS60qnc7m76/TiylYXH9Op7PFu7v5n9n801els50iKxGG0pmlcmZUiw7dUeT7Tss6altbhy7mu4fyK2nag59vgPr4JWJ9EnMQYl/YUC/VIGD55SfHgg7kqAJBjwpBo1UUYORhlkUWBD6lYh3Iy0KgXohPERMcuHQYZ3ep3O96HrAG8Wx45qNqTe8Z1jHq4bJfrcHlY8H+57Ggvk9c3KHuMMKxyF8LhkMkCI15QAZ8dfdstiPD9cvk/AVQSwMEFAAAAAgAgbT8XHSoGMk8AQAAYgIAABQAAAB4bC90YWJsZXMvdGFibGUxLnhtbHWRz07CQBCHX2Wzd2kBMaahEEM0IVETxRdY6ZRusn+anamUm0cMNxMTL8Z4U08efSbAdzAUqaBw/eab2f3NNNu5VuwGHEprQl6t+JyB6dtImkHIM4r3Dnm71cwDEtcKmIxCXuXMCA0hv8gsCZLWdAk0chZJTJUYnW8tOohDflQNjuucJSAicJd22LGZoWIiWRIKf5G/hnqJHZoC5VoZDPKQJ0Rp4HnYT0ALrNgUTK5VbJ0WhBXrBh6mDkSECQBp5dV8/8DTQhpeZulYlWmDrL98sPG3spn16348e3/jzNtm1VbWfPw4e77bYdVX1mzyOv243WHtr82aT552WI2VNf0czx9eCsvbDFa29WikoGti+9NyVcIziGSma5xhYocn0iEte4tdL9ip+IcW9yAnU8BiOwu0NErqb/ym9Q1QSwMEFAAAAAgAgbT8XOeqHG/LAgAAsAYAABgAAAB4bC93b3Jrc2hlZXRzL3NoZWV0My54bWyVle9P2kAYx/+VJ/d6o+VHN0ME43RuJluy7MX2utIDGmmPtKfwkkxmnLhEN4noWqKJi+JMxgQzXPAfap/+D0sLIi50c6+4566f792HKw/TM2WtAKvUMFWmp0g0IhKgeoYpqp5LkRWefThFZtLT5WSJGctmnlIOZa2gm8lyiuQ5LyYFwczkqSabEVakelkrZJmhydyMMCMnmEWDykqAaQUhJoqPBE1WdeIHBrNvVFoy71Rg5lnpmaEqL1SdmikiEvC3XmJs2V9eVIIpIT0tTIxYCHZ/ZYBCs/JKgb9mpedUzeV5ikSlgCsnM6wQABlWAE31rQlocjn4LKkKz6dIdIpAXlUUqgfbZVZMzrS3w7XbmAEeG+KxEZ6I/QceH+LxER4X/4ULtxaB9rzMZb8wWAkM/6FgB384GyVgDiJ5ipjcCJZW016jjjUb8GAX17f8vNVB6gh8EgLOs8yKRnUOs4tOuwLu+me8aji9NuCHczzognOxNiluLiQOa8e4eQnefh3t3h1QCHTGrGJjVrFBWvTPtI0G1uyJOiGEt2NhzXLan9yTLUD7Gq2+2+qC06niuzY4P/vOj7b7tQ/YXEN74hc1F5L8NAb4vgKDI91LMD4mGJ8c6nZ2vHrLD54oGULhoeWetdCugvuxjXbXq7YB97rukQXYOEa759UP/YMCNjfCLMOiD3axfg2z0eRC9F6SiTHJRNgtnjpXFcCa5W32JnqGgN7+rn95+OsILxq4tw1up4t2F08qgPXNMLOwsEYVm+fgrV969ZZ73sOqNTwSuGct9+h+dyqN6Uphut+wuQ1YtcLe3RBQkkRRFJ1OHyRJfCCKIn7ZRrsBaDfc7z3A9ina3YnKIYELCRj8HmDYHv5ieNN/b1pPUc7Rl7KRU3UTCjTLU0SMPCZgDJpvMOasGIwkAkuMc6bdVHkqK9TwqziBLGN8VAx63ejPJ/0bUEsDBBQAAAAAAIG0/Fx9PCR5KAEAACgBAAALAAAAX3JlbHMvLnJlbHPvu788P3htbCB2ZXJzaW9uPSIxLjAiIGVuY29kaW5nPSJ1dGYtOCI/PjxSZWxhdGlvbnNoaXBzIHhtbG5zPSJodHRwOi8vc2NoZW1hcy5vcGVueG1sZm9ybWF0cy5vcmcvcGFja2FnZS8yMDA2L3JlbGF0aW9uc2hpcHMiPjxSZWxhdGlvbnNoaXAgVHlwZT0iaHR0cDovL3NjaGVtYXMub3BlbnhtbGZvcm1hdHMub3JnL29mZmljZURvY3VtZW50LzIwMDYvcmVsYXRpb25zaGlwcy9vZmZpY2VEb2N1bWVudCIgVGFyZ2V0PSIveGwvd29ya2Jvb2sueG1sIiBJZD0iUmUwODM1NWJhNGQxZjQ4MDgiIC8+PC9SZWxhdGlvbnNoaXBzPlBLAwQUAAAACACBtPxcJDEdJzIBAAAwBAAAGgAAAHhsL19yZWxzL3dvcmtib29rLnhtbC5yZWxzzdNLTsMwEAbgq0TeE9uxmwdq2g0btqUXmNqThxrbke1CejYWHIkrIApCCWLBplI3s/hH+vV5JL+/vq23kxmSZ/Shd7YmPGUkQauc7m1bk1Ns7kqy3ax3OEDsnQ1dP4ZkMoMNNeliHO8pDapDAyF1I9rJDI3zBmJInW/pCOoILdKMsZz6eQdZdib784j/aXRN0yt8cOpk0MY/immI5wEDSfbgW4w1odPwnaWTGUjyqGuy40WpilKvoCpzmXNNEno1UOzQ4NJzib4mn6lKniuWA4AoSpnl6pqq0IFH/RR9b9vf15qvZrwVE0VWMYWZWEkt+TV5L84fQ4cYl7Sf+PMBiHF+vUYIUAUTUjVaVoW8AV624B2qCnWOAg+ygPIGeGLGk5qzjGuVNXCQsMILjy7+/eYDUEsDBBQAAAAIAIG0/FxV2/KPuwAAACQBAAAjAAAAeGwvd29ya3NoZWV0cy9fcmVscy9zaGVldDIueG1sLnJlbHONzztuwzAQRdGtENNbIyWB/IEoN2nSGt7AiB5KhPkDSQfM2lJkSdmCC7uwgRRp3wUO8H6/f4Z9dVZ8csomeAld04Jgr8LJ+FnCpejVBvbjcGBLxQSfFxOzqM76LGEpJe4Qs1rYUW5CZF+d1SE5KrkJacZI6kwz40vb9pgeDXg2xfEr8n/EoLVR/B7UxbEvf8BYaLIM4khp5iIBq71N99I11VkQHycJB039up261y1N/Kb7NQgcB3z6Ol4BUEsDBBQAAAAIAIG0/FySa1z3KwEAAOYEAAATAAAAW0NvbnRlbnRfVHlwZXNdLnhtbMWUTU7DMBBGrxJ5i2q3RUIINe0C2AISXGBwJolV/8kzDenZWHAkroDqoAohpKgiiI1nY7/3fbPw++vbatM7W3SYyARfioWciwK9DpXxTSl2XM8uxWa9etpHpKJ31lMpWuZ4pRTpFh2QDBF972wdkgMmGVKjIugtNKiW8/mF0sEzep7xgSHWqxusYWe5uO0Z/aDtnRXF9XDvoCoFxGiNBjbBq85X3ySzUNdGYxX0zqFnSTEhVNQisrMyT+nA+LMMVj86E1o6TfrZSia0+Q61JtJRcd9hSqbC4gES34HDUqjeKuK9RZITN8zQMTW36HA4F78OkDGjZVtIWD1yMr6ZvPNX9liQl5C2+SGpPBYThznyTw2y/K8gDM8WaRhTbyNDT93E+Z9vQuVfa/0BUEsBAhQDFAAAAAgAgbT8XE99g3MdAQAAUAIAAA8AAAAAAAAAAAAAAKSBAAAAAHhsL3dvcmtib29rLnhtbFBLAQIUAxQAAAAIAIG0/Fyo7/cH1wIAACMbAAANAAAAAAAAAAAAAACkgUoBAAB4bC9zdHlsZXMueG1sUEsBAhQDFAAAAAgAgbT8XPpcAVkDAwAA2g0AABMAAAAAAAAAAAAAAKSBTAQAAHhsL3RoZW1lL3RoZW1lMS54bWxQSwECFAMUAAAACACBtPxcDR656GUAAABzAAAAFAAAAAAAAAAAAAAApIGABwAAeGwvc2hhcmVkU3RyaW5ncy54bWxQSwECFAMUAAAACACBtPxce2eF/OQDAAB9DAAAGAAAAAAAAAAAAAAApIEXCAAAeGwvd29ya3NoZWV0cy9zaGVldDEueG1sUEsBAhQDFAAAAAgAgbT8XMl9569uAgAAzQYAABgAAAAAAAAAAAAAAKSBMQwAAHhsL3dvcmtzaGVldHMvc2hlZXQyLnhtbFBLAQIUAxQAAAAIAIG0/Fx0qBjJPAEAAGICAAAUAAAAAAAAAAAAAACkgdUOAAB4bC90YWJsZXMvdGFibGUxLnhtbFBLAQIUAxQAAAAIAIG0/FznqhxvywIAALAGAAAYAAAAAAAAAAAAAACkgUMQAAB4bC93b3Jrc2hlZXRzL3NoZWV0My54bWxQSwECFAMUAAAAAACBtPxcfTwkeSgBAAAoAQAACwAAAAAAAAAAAAAApIFEEwAAX3JlbHMvLnJlbHNQSwECFAMUAAAACACBtPxcJDEdJzIBAAAwBAAAGgAAAAAAAAAAAAAApIGVFAAAeGwvX3JlbHMvd29ya2Jvb2sueG1sLnJlbHNQSwECFAMUAAAACACBtPxcVdvyj7sAAAAkAQAAIwAAAAAAAAAAAAAApIH/FQAAeGwvd29ya3NoZWV0cy9fcmVscy9zaGVldDIueG1sLnJlbHNQSwECFAMUAAAACACBtPxckmtc9ysBAADmBAAAEwAAAAAAAAAAAAAApIH7FgAAW0NvbnRlbnRfVHlwZXNdLnhtbFBLBQYAAAAADAAMACIDAABXGAAAAAA=', 'application_form.docx': '...', 'transaction_statement.pdf': '...', 'table_summary.pptx': '...'}
office_dir = OUTPUT_DIR / "office_format_samples"
office_dir.mkdir(exist_ok=True)
for filename, payload in OFFICE_FILES.items():
    (office_dir / filename).write_bytes(base64.b64decode(payload))

def xml_text_count(path, prefix, text_tag):
    with zipfile.ZipFile(path) as archive:
        names = [
            name for name in archive.namelist()
            if name.startswith(prefix) and name.endswith(".xml")
        ]
        text_count = 0
        for name in names:
            xml = archive.read(name).decode("utf-8", errors="ignore")
            text_count += len(re.findall(text_tag, xml))
        return len(names), text_count

xlsx_sheets, xlsx_values = xml_text_count(
    office_dir / "quotation.xlsx",
    "xl/worksheets/",
    r"<x:(?:v|f)>",
)
docx_parts, docx_text = xml_text_count(
    office_dir / "application_form.docx",
    "word/document",
    r"<w:t",
)
pptx_slides, pptx_text = xml_text_count(
    office_dir / "table_summary.pptx",
    "ppt/slides/slide",
    r"<a:t>",
)
pdf_bytes = (office_dir / "transaction_statement.pdf").read_bytes()
print("Excel:", xlsx_sheets, "개 시트 XML · 값/수식", xlsx_values)
print("Word:", docx_text, "개 본문 텍스트 run · 이미지 본문 여부 확인")
print("PDF:", pdf_bytes[:5], "· 텍스트층 샘플")
print("PPT:", pptx_slides, "개 슬라이드 · 텍스트", pptx_text)

office_bundle = OUTPUT_DIR / "office_format_samples.zip"
with zipfile.ZipFile(office_bundle, "w") as archive:
    for path in sorted(office_dir.iterdir()):
        archive.write(path, path.name)
print("실제 파일 4종 묶음:", office_bundle)
download_artifact(office_bundle)


## 내가 직접 만드는 PoC 카드

`candidate`는 `quotation`, `application`, `transaction_statement`
중 하나입니다. 점수는 1~5점이며 오류 영향과 예외 빈도는 낮을수록
첫 PoC에 유리합니다.


In [ ]:
# TODO: 내 업무 후보와 점수·검토자·중단 조건을 채우세요.
candidate = None
score = {
    "반복량": None,
    "필드 안정성": None,
    "오류 영향": None,
    "예외 빈도": None,
    "사람 검토 가능성": None,
}
review_owner = None
stop_condition = None
if candidate is None or any(value is None for value in score.values()):
    print("빈칸이 있습니다. 아래 힌트·전체 정답과 비교하세요.")


<details>
<summary>힌트와 전체 정답 보기</summary>

예시는 거래명세서를 한 장씩 처리하고 정산 담당자가 검토하는 작은
PoC입니다. 값이 맞지 않거나 원본 근거가 없으면 저장을 중단합니다.
</details>


In [ ]:
from textwrap import dedent

candidate = candidate or "transaction_statement"
if candidate not in EXTENSION_EXAMPLES:
    raise ValueError(
        "candidate는 quotation, application, "
        "transaction_statement 중 하나여야 합니다."
    )
default_score = {
    "반복량": 4,
    "필드 안정성": 4,
    "오류 영향": 2,
    "예외 빈도": 3,
    "사람 검토 가능성": 5,
}
score = {
    key: (
        int(value)
        if value is not None
        else default_score[key]
    )
    for key, value in score.items()
}
if not all(1 <= value <= 5 for value in score.values()):
    raise ValueError("모든 점수는 1~5 사이여야 합니다.")
review_owner = review_owner or "정산 담당자"
stop_condition = (
    stop_condition
    or "필수값·합계·원본 근거 중 하나라도 틀리면 자동 저장 중단"
)
example = EXTENSION_EXAMPLES[candidate]
recommendation = (
    "GO_SMALL"
    if (
        score["반복량"] >= 4
        and score["필드 안정성"] >= 3
        and score["오류 영향"] <= 3
        and score["예외 빈도"] <= 3
        and score["사람 검토 가능성"] >= 4
    )
    else "REVIEW"
)
card = f'''# 문서 자동화 PoC 후보 카드

| 항목 | 내용 |
| --- | --- |
| 선택 문서 | {example["name"]} |
| 추출 필드 | {", ".join(example["fields"])} |
| 검증 규칙 | {" / ".join(example["rules"])} |
| 틀렸을 때 영향 | {example["risk"]} |
| 입력 제한 | 승인된 비식별 한 장 |
| 최종 산출물 | 사람 승인 후 Excel |
| 사람 검토자 | {review_owner} |
| 중단 조건 | {stop_condition} |
| 점수 | {" / ".join(f"{key} {value}" for key, value in score.items())} |
| 제안 | {recommendation} |

## 첫 PoC 통과 기준

- 같은 양식 30장을 모아 정답표와 비교한다.
- 필드별 정확도뿐 아니라 수정률과 처리시간을 기록한다.
- 오류 시 자동 저장하지 않고 검토 대기열로 보낸다.
- 개인정보·보존·삭제 정책을 먼저 승인받는다.
'''
output_path = OUTPUT_DIR / "poc_candidate_card.md"
output_path.write_text(dedent(card), encoding="utf-8")
print(dedent(card))
print("CHECKPOINT 1/1 PASS:", output_path)
download_artifact(output_path)
